# CodeX Phase 1 - Survey Data Cleaning


This notebook applies the cleaning instructions from `data_cleaning_instructions.pdf` to `datasets/survey_results.csv`.


In [1]:
import pandas as pd


In [2]:
raw_path = 'dataset/survey_results.csv'
clean_path = 'dataset/survey_results_cleaned.csv'

df = pd.read_csv(raw_path)
print('Raw shape:', df.shape)
df.head()


Raw shape: (30010, 17)


,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200
4,R00005,23,M,Metro,Student,NaN,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100


## 1) Remove Duplicates


In [3]:
print('Duplicate respondent_id before:', df['respondent_id'].duplicated().sum())
print('Exact duplicate rows before:', df.duplicated().sum())


Duplicate respondent_id before: 10
Exact duplicate rows before: 10


In [4]:
# Normalize whitespace for text fields first
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip()
    df.loc[df[col].isin(['nan', 'None']), col] = pd.NA

# Use respondent_id as business key for deduplication
# (same respondent should appear only once)
df = df.drop_duplicates(subset=['respondent_id'], keep='first').copy()

print('Shape after dedup:', df.shape)
print('Duplicate respondent_id after:', df['respondent_id'].duplicated().sum())


Shape after dedup: (30000, 17)
Duplicate respondent_id after: 0


## 2) Outlier Detection in Age


In [5]:
q1 = df['age'].quantile(0.25)
q3 = df['age'].quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print('IQR bounds:', low, high)
print('IQR outliers:', ((df['age'] < low) | (df['age'] > high)).sum())
print('Age min/max before age-fix:', df['age'].min(), df['age'].max())


IQR bounds: -2.5 65.5
IQR outliers: 493
Age min/max before age-fix: 18 604


In [6]:
# Remove clearly invalid age entries (>100)
# Keep valid senior ages <=100 even if IQR flags them.
df = df[df['age'] <= 100].copy()
print('Age min/max after age-fix:', df['age'].min(), df['age'].max())


Age min/max after age-fix: 18 70


## 3) Handle Missing Data


In [7]:
print('Missing before:')
print(df[['income_levels', 'consume_frequency(weekly)', 'purchase_channel']].isna().sum())
# As per instruction hint
df['income_levels'] = df['income_levels'].fillna('Not Reported')
# Mode imputation for low-missing categorical columns
consume_mode = df['consume_frequency(weekly)'].mode().iat[0]
purchase_mode = df['purchase_channel'].mode().iat[0]
df['consume_frequency(weekly)'] = df['consume_frequency(weekly)'].fillna(consume_mode)
df['purchase_channel'] = df['purchase_channel'].fillna(purchase_mode)
print()
print('Imputation modes used:')
print('consume_frequency(weekly):', consume_mode)
print('purchase_channel:', purchase_mode)
print()
print('Missing after:')
print(df[['income_levels', 'consume_frequency(weekly)', 'purchase_channel']].isna().sum())


Missing before:
income_levels                8060
consume_frequency(weekly)       8
purchase_channel               10
dtype: int64

Imputation modes used:
consume_frequency(weekly): 3-4 times
purchase_channel: Online

Missing after:
income_levels                0
consume_frequency(weekly)    0
purchase_channel             0
dtype: int64


## 4) Correct Spelling Mistakes in Categorical Columns


In [8]:
print('Zone values before fix:', sorted(df['zone'].dropna().unique().tolist()))
print('Current brand values before fix:', sorted(df['current_brand'].dropna().unique().tolist()))


Zone values before fix: ['Metor', 'Metro', 'Rural', 'Semi-Urban', 'Urban', 'urbna']
Current brand values before fix: ['Establishd', 'Established', 'Newcomer', 'newcomer']


In [9]:
zone_map = {'Metor': 'Metro', 'urbna': 'Urban'}
brand_map = {'Establishd': 'Established', 'newcomer': 'Newcomer'}

df['zone'] = df['zone'].replace(zone_map)
df['current_brand'] = df['current_brand'].replace(brand_map)

print('Zone values after fix:', sorted(df['zone'].dropna().unique().tolist()))
print('Current brand values after fix:', sorted(df['current_brand'].dropna().unique().tolist()))


Zone values after fix: ['Metro', 'Rural', 'Semi-Urban', 'Urban']
Current brand values after fix: ['Established', 'Newcomer']


## Final Validation and Export


In [10]:
validation = {
    'rows_after_cleaning': int(df.shape[0]),
    'columns': int(df.shape[1]),
    'duplicate_respondent_id': int(df['respondent_id'].duplicated().sum()),
    'exact_duplicate_rows': int(df.duplicated().sum()),
    'age_gt_100': int((df['age'] > 100).sum()),
    'income_levels_missing': int(df['income_levels'].isna().sum()),
    'consume_frequency_missing': int(df['consume_frequency(weekly)'].isna().sum()),
    'purchase_channel_missing': int(df['purchase_channel'].isna().sum())
}
validation


{'rows_after_cleaning': 29991,
 'columns': 17,
 'duplicate_respondent_id': 0,
 'exact_duplicate_rows': 0,
 'age_gt_100': 0,
 'income_levels_missing': 0,
 'consume_frequency_missing': 0,
 'purchase_channel_missing': 0}

In [11]:
df.to_csv(clean_path, index=False)
print('Saved cleaned dataset to:', clean_path)

Saved cleaned dataset to: datasets/survey_results_cleaned.csv
